# Séance 4 — Généralisation, Validation & Prétraitement (CORRIGÉ)

**Formation ML — formation-ml**

### Objectifs

- Comprendre pourquoi l'erreur d'entraînement est un indicateur trompeur (sous/sur-apprentissage)
- Retrouver numériquement la décomposition biais-variance
- Mesurer l'instabilité d'un split unique et la corriger avec la validation croisée
- Construire Ridge et Lasso, comparer leurs effets, choisir λ par validation croisée
- Nettoyer et préparer Titanic : valeurs manquantes, encodage, mise à l'échelle
- Provoquer puis corriger une fuite de données (data leakage)
- Assembler tout ça dans un `Pipeline` scikit-learn

*Deux datasets aujourd'hui : California Housing (Partie A, régression) et Titanic (Partie B, prétraitement).*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, accuracy_score

sns.set_theme(style="whitegrid")
np.random.seed(42)

## Partie A — Généraliser et valider

### 1. Le piège de l'erreur d'entraînement

In [ ]:
housing = fetch_california_housing(as_frame=True)
df_h = housing.frame
df_h.head()

On se concentre sur une seule feature, `Latitude`, contre `MedHouseVal` — la relation n'est pas une droite (villes côtières chères des deux côtés de l'État), ce qui va nous permettre de voir clairement le sous- et le sur-apprentissage.

<div style="background:#eef6ff;border-left:5px solid #1a73e8;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>💡 Intuition</b><br>
Avec un petit échantillon (35 points) et un polynôme de degré 14, le modèle a largement assez de paramètres pour <b>mémoriser</b> chaque point. Regarde ce que ça donne hors de l'échantillon d'entraînement.
</div>

In [ ]:
rng = np.random.default_rng(0)
sample_idx = rng.choice(len(df_h), size=35, replace=False)
X_lat = df_h["Latitude"].values[sample_idx].reshape(-1, 1)
y_lat = df_h["MedHouseVal"].values[sample_idx]

x_plot = np.linspace(df_h["Latitude"].min(), df_h["Latitude"].max(), 300).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, degree in zip(axes, [1, 3, 14]):
    coefs = np.polyfit(X_lat.flatten(), y_lat, degree)
    y_fit = np.polyval(coefs, x_plot.flatten())
    y_train_pred = np.polyval(coefs, X_lat.flatten())
    mse_train = mean_squared_error(y_lat, y_train_pred)
    ax.scatter(X_lat, y_lat, alpha=0.6, s=25)
    ax.plot(x_plot, y_fit, color="crimson", linewidth=2)
    ax.set_ylim(0, 5.5)
    ax.set_title(f"degré {degree}  (MSE train={mse_train:.3f})")
    ax.set_xlabel("Latitude")
axes[0].set_ylabel("MedHouseVal")
plt.tight_layout()
plt.show()

<div style="background:#f3e5f5;border-left:5px solid #8e24aa;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>✏️ Exercice</b><br>
Le degré 14 a le MSE d'entraînement le plus bas des trois. Est-ce le modèle que tu choisirais pour prédire sur de <b>nouveaux</b> districts ? Justifie en une phrase avant de continuer.
</div>

### 2. La vraie mesure : erreur d'entraînement **et** de test

In [ ]:
X_full = df_h[["Latitude"]].values
y_full = df_h["MedHouseVal"].values
X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, train_size=200, test_size=4000, random_state=0)

degrees = range(1, 13)
train_errors, test_errors = [], []
for d in degrees:
    coefs = np.polyfit(X_train.flatten(), y_train, d)
    train_errors.append(mean_squared_error(y_train, np.polyval(coefs, X_train.flatten())))
    test_errors.append(mean_squared_error(y_test, np.polyval(coefs, X_test.flatten())))

plt.figure(figsize=(7, 5))
plt.plot(list(degrees), train_errors, marker="o", label="Erreur d'entraînement")
plt.plot(list(degrees), test_errors, marker="o", label="Erreur de test")
plt.xlabel("Degré du polynôme (complexité)")
plt.ylabel("MSE")
plt.legend()
plt.title("Le compromis biais-variance, en direct")
plt.show()

print("Meilleur degré (test) :", list(degrees)[int(np.argmin(test_errors))])

<div style="background:#fff3e0;border-left:5px solid #e67e22;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>⚠️ Lecture</b><br>
L'erreur d'entraînement décroît toujours. L'erreur de test descend puis remonte : c'est la signature du compromis biais-variance. Le minimum dépend de la <b>quantité de données disponible</b> — avec plus de données, le minimum se déplacerait vers la droite.
</div>

### 3. Vérifier la décomposition biais-variance numériquement

On rééchantillonne 40 fois un même petit jeu d'entraînement (tirages frais dans la population complète), on ajuste un degré 1 et un degré 14 à chaque fois, et on regarde la dispersion des courbes obtenues.

In [ ]:
# Latitude standardisée + grille restreinte au 1er-99e percentile : un polynôme de
# degré 14 ajusté sur seulement 40 points diverge très vite en extrapolation (phénomène
# de Runge) — on borne les prédictions pour rester dans une échelle lisible et honnête.
lat_std = (df_h["Latitude"] - df_h["Latitude"].mean()) / df_h["Latitude"].std()

def fit_many(degree, n_resamples=40, n_train=40, seed=0):
    rng = np.random.default_rng(seed)
    x_grid = np.linspace(np.percentile(lat_std, 1), np.percentile(lat_std, 99), 200)
    preds = np.zeros((n_resamples, len(x_grid)))
    for i in range(n_resamples):
        idx = rng.choice(len(df_h), size=n_train, replace=False)
        coefs = np.polyfit(lat_std.values[idx], df_h["MedHouseVal"].values[idx], degree)
        preds[i] = np.clip(np.polyval(coefs, x_grid), -1, 6)
    return x_grid, preds

x_grid, preds_deg1 = fit_many(1)
_, preds_deg14 = fit_many(14)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, preds, title in zip(axes, [preds_deg1, preds_deg14], ["degré 1", "degré 14"]):
    for p in preds:
        ax.plot(x_grid, p, color="steelblue", alpha=0.15)
    ax.plot(x_grid, preds.mean(axis=0), color="crimson", linewidth=2, label="moyenne sur les 40 tirages")
    ax.set_ylim(-0.5, 6)
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()

print("Variance moyenne (degré 1) :", preds_deg1.var(axis=0).mean().round(4))
print("Variance moyenne (degré 14):", preds_deg14.var(axis=0).mean().round(4))

<div style="background:#e8f5e9;border-left:5px solid #2e7d32;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>✅ À retenir</b><br>
Degré 1 : les 40 courbes sont presque superposées (variance faible) mais aucune ne suit vraiment la tendance réelle (biais élevé). Degré 14 : les courbes divergent fortement (variance élevée) mais leur <b>moyenne</b> suit mieux la tendance (biais plus faible). C'est exactement Biais² + Variance.
</div>

### 4. Validation croisée : un split suffit-il ?

In [ ]:
X_reg = df_h[["MedInc", "AveRooms", "AveBedrms", "HouseAge"]].values[:400]
y_reg = df_h["MedHouseVal"].values[:400]

# 30 splits uniques, seeds différentes
single_split_mse = []
for seed in range(30):
    Xtr, Xte, ytr, yte = train_test_split(X_reg, y_reg, test_size=0.2, random_state=seed)
    model = Ridge(alpha=1.0).fit(Xtr, ytr)
    single_split_mse.append(mean_squared_error(yte, model.predict(Xte)))
single_split_mse = np.array(single_split_mse)

# Une seule validation croisée à 5 plis
kf = KFold(n_splits=5, shuffle=True, random_state=0)
cv_scores = -cross_val_score(Ridge(alpha=1.0), X_reg, y_reg, cv=kf, scoring="neg_mean_squared_error")

print(f"30 splits uniques : moyenne={single_split_mse.mean():.3f}, écart-type={single_split_mse.std():.3f}")
print(f"5-fold CV         : moyenne={cv_scores.mean():.3f}, écart-type={cv_scores.std():.3f}")

<div style="background:#f3e5f5;border-left:5px solid #8e24aa;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>✏️ Exercice</b><br>
Refais le calcul avec <code>KFold(n_splits=10, ...)</code>. L'écart-type diminue-t-il encore ? Est-ce gratuit (réfléchis au temps de calcul) ?
</div>

### 5. Régularisation : Ridge et Lasso

`AveRooms` et `AveBedrms` sont fortement corrélées (0.85) — un terrain propice à l'instabilité des moindres carrés ordinaires (OLS).

In [ ]:
X_corr = df_h[["AveRooms", "AveBedrms"]].values
y_corr = df_h["MedHouseVal"].values

ols = LinearRegression().fit(X_corr, y_corr)
ridge = Ridge(alpha=1.0).fit(X_corr, y_corr)
lasso = Lasso(alpha=0.01).fit(X_corr, y_corr)

print("OLS   :", ols.coef_.round(3))
print("Ridge :", ridge.coef_.round(3))
print("Lasso :", lasso.coef_.round(3))

<div style="background:#eef6ff;border-left:5px solid #1a73e8;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>💡 Intuition</b><br>
L'OLS donne des coefficients de <b>signes opposés</b> qui se compensent artificiellement — un signe d'instabilité classique en présence de colinéarité. Ridge réduit les deux en douceur. Lasso peut carrément éliminer une variable (coefficient à 0).
</div>

**Chemin de régularisation** : on fait varier λ et on observe l'évolution de chaque coefficient.

In [ ]:
features_all = ["MedInc", "HouseAge", "AveRooms", "AveBedrms", "Population", "AveOccup", "Latitude", "Longitude"]
X_all = df_h[features_all].values
y_all = df_h["MedHouseVal"].values
X_all_scaled = StandardScaler().fit_transform(X_all)

alphas = np.logspace(-2, 3, 40)
coefs_path = np.array([Ridge(alpha=a).fit(X_all_scaled, y_all).coef_ for a in alphas])

plt.figure(figsize=(8, 5))
for j, feat in enumerate(features_all):
    plt.plot(alphas, coefs_path[:, j], label=feat)
plt.xscale("log")
plt.xlabel("λ (log)")
plt.ylabel("coefficient")
plt.title("Chemin de régularisation — Ridge")
plt.legend(fontsize=8)
plt.show()

### 6. Choisir λ par validation croisée

In [ ]:
param_grid = {"alpha": np.logspace(-3, 3, 20)}
features_all = ["MedInc", "HouseAge", "AveRooms", "AveBedrms", "Population", "AveOccup", "Latitude", "Longitude"]
X_all_scaled = StandardScaler().fit_transform(df_h[features_all].values)
y_all = df_h["MedHouseVal"].values

grid = GridSearchCV(Ridge(), param_grid, cv=5, scoring="neg_mean_squared_error")
grid.fit(X_all_scaled, y_all)

print("Meilleur alpha :", grid.best_params_["alpha"])
print("MSE (CV) associé :", -grid.best_score_)

<div style="background:#fff3e0;border-left:5px solid #e67e22;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>⚠️ Piège à éviter</b><br>
Ce <code>GridSearchCV</code> choisit λ par validation croisée sur les données disponibles — jamais sur un jeu de test qu'on garderait ensuite pour l'évaluation finale. Le test ne doit servir <b>qu'une fois</b>, à la toute fin.
</div>

## Partie B — Préparer les données correctement

On repasse sur Titanic pour la partie prétraitement — assure-toi d'avoir `titanic.csv` dans ce dossier.

### 6. Valeurs manquantes

In [ ]:
df_t = pd.read_csv("titanic.csv")
df_t.isna().mean().sort_values(ascending=False).head(5) * 100

<div style="background:#eef6ff;border-left:5px solid #1a73e8;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>💡 Intuition</b><br>
<code>Cabin</code> manque à plus de 77% — difficile d'en tirer grand-chose telle quelle. <code>Age</code> manque à ~20% — trop pour être ignorée, mais imputable. <code>Embarked</code> ne manque qu'à 0.2% — négligeable.
</div>

In [ ]:
num_imputer = SimpleImputer(strategy="median")
df_t["Age_imputed"] = num_imputer.fit_transform(df_t[["Age"]])

cat_imputer = SimpleImputer(strategy="most_frequent")
df_t["Embarked_imputed"] = cat_imputer.fit_transform(df_t[["Embarked"]]).ravel()

print("Age manquants avant/après :", df_t["Age"].isna().sum(), "/", df_t["Age_imputed"].isna().sum())

### 7. Encoder les variables catégorielles

In [ ]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
embarked_encoded = encoder.fit_transform(df_t[["Embarked_imputed"]])
pd.DataFrame(embarked_encoded, columns=encoder.get_feature_names_out()).head()

<div style="background:#fff3e0;border-left:5px solid #e67e22;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>⚠️ Piège</b><br>
Un encodage ordinal naïf (<code>S→0, C→1, Q→2</code>) affirmerait un ordre numérique qui n'existe pas pour un port d'embarquement. L'encodage ordinal reste légitime pour <code>Pclass</code>, qui a un vrai ordre socio-économique.
</div>

### 8. Mise à l'échelle : pourquoi c'est loin d'être cosmétique

In [ ]:
# Fare (0-512) et une copie à échelle 100x plus petite : EXACTEMENT la même information
fare = df_t["Fare"].values
X_dup = np.column_stack([fare, fare / 100.0])
y_bin = df_t["Survived"].values

ridge_raw = Ridge(alpha=1.0).fit(X_dup, y_bin)
ridge_scaled = Ridge(alpha=1.0).fit(StandardScaler().fit_transform(X_dup), y_bin)

print("Sans standardisation -> |coef| :", np.abs(ridge_raw.coef_).round(6))
print("Avec standardisation -> |coef| :", np.abs(ridge_scaled.coef_).round(4))

<div style="background:#e8f5e9;border-left:5px solid #2e7d32;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>✅ À retenir</b><br>
Les deux colonnes portent la <b>même information</b>, juste à des échelles différentes. Sans standardisation, la régularisation les traite très différemment — un pur artefact d'unité. Avec standardisation, la pénalité est répartie équitablement.
</div>

### 9. La fuite de données (data leakage)

`Ticket` a 681 valeurs uniques pour 891 lignes. On l'encode par la moyenne de `Survived` par valeur de ticket — avec et sans fuite.

In [ ]:
y_leak = df_t["Survived"].values
n = len(df_t)
kf5 = KFold(n_splits=5, shuffle=True, random_state=0)

# Avec fuite : moyenne calculée sur TOUT le dataset avant la CV
enc_leak = df_t.groupby("Ticket")["Survived"].transform("mean").values.reshape(-1, 1)

# Sans fuite : moyenne recalculée à chaque pli, uniquement sur le train
def encode_clean(train_idx, all_idx):
    d_train = df_t.iloc[train_idx]
    global_mean = d_train["Survived"].mean()
    ticket_means = d_train.groupby("Ticket")["Survived"].mean()
    enc = np.full(n, global_mean)
    for i in all_idx:
        t = df_t.iloc[i]["Ticket"]
        enc[i] = ticket_means[t] if t in ticket_means.index else global_mean
    return enc

leak_acc, clean_acc = [], []
for tr, te in kf5.split(df_t):
    m1 = LogisticRegression().fit(enc_leak[tr], y_leak[tr])
    leak_acc.append(accuracy_score(y_leak[te], m1.predict(enc_leak[te])))
    enc_clean = encode_clean(tr, list(tr) + list(te)).reshape(-1, 1)
    m2 = LogisticRegression().fit(enc_clean[tr], y_leak[tr])
    clean_acc.append(accuracy_score(y_leak[te], m2.predict(enc_clean[te])))

baseline = max(y_leak.mean(), 1 - y_leak.mean())
print(f"Baseline (classe majoritaire) : {baseline:.1%}")
print(f"Encodage correct (par pli)    : {np.mean(clean_acc):.1%}")
print(f"Encodage avec fuite           : {np.mean(leak_acc):.1%}")

<div style="background:#e8f5e9;border-left:5px solid #2e7d32;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>✅ À retenir</b><br>
L'encodage avec fuite affiche une accuracy spectaculaire mais fictive — un ticket rare porte presque sa propre étiquette. Seule la version sans fuite se généraliserait réellement en production.
</div>

### 10. Tout assembler avec un `Pipeline`

In [ ]:
num_features = ["Age", "Fare"]
cat_features = ["Sex", "Embarked"]

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features),
])

model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression()),
])

X = df_t[num_features + cat_features]
y = df_t["Survived"]

scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
print(f"Accuracy (CV) : {scores.mean():.3f} +/- {scores.std():.3f}")

<div style="background:#e8f5e9;border-left:5px solid #2e7d32;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>✅ À retenir</b><br>
<code>cross_val_score(model, ...)</code> réajuste <b>tout le pipeline</b> — imputation, encodage, scaling, modèle — séparément à chaque pli. La fuite de données devient structurellement impossible, sans y penser manuellement à chaque étape.
</div>

In [ ]:
param_grid_pipe = {"classifier__C": np.logspace(-3, 2, 12)}
grid_pipe = GridSearchCV(model, param_grid_pipe, cv=5, scoring="accuracy")
grid_pipe.fit(X, y)

print("Meilleur C :", grid_pipe.best_params_["classifier__C"])
print("Accuracy (CV) associée :", grid_pipe.best_score_.round(3))

## Débrief

| Brique | Séances 2-3 | Séance 4 |
|---|---|---|
| ① Modèle | Droite, sigmoïde | + un curseur de complexité (degré, nb de variables) |
| ② Coût | MSE, log-loss | + un terme de pénalité (Ridge/Lasso) |
| ③ Optimisation | Descente de gradient | + un second niveau : hyperparamètres choisis par CV |

Et une étape 0, nouvelle aujourd'hui : préparer les données (valeurs manquantes, encodage, échelle), dans un `Pipeline`, pour ne jamais laisser fuiter d'information du test vers l'entraînement.

## Auto-évaluation

1. Pourquoi un modèle avec une erreur d'entraînement quasi nulle peut-il être un **mauvais** modèle ?
2. Dans la décomposition biais-variance, que se passe-t-il pour le biais et la variance quand la complexité du modèle augmente ?
3. Pourquoi l'écart-type d'un score de validation croisée est-il un meilleur indicateur de fiabilité qu'un seul split train/test ?
4. Pourquoi Lasso peut-il mettre un coefficient exactement à zéro, contrairement à Ridge ?
5. Donne un exemple concret de fuite de données autre que celui vu dans ce notebook.
6. Pourquoi un `Pipeline` réduit-il structurellement le risque de fuite, par rapport à des transformations appliquées "à la main" ?